In [97]:
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
load_dotenv()
client = OpenAI()

In [98]:
sentence_1 = "Forxiga caused UTI in 8.4% of patients in DECLARE-TIMI 58"
resp = client.embeddings.create(
    model="text-embedding-3-small",
    input=sentence_1
)

vector = resp.data[0].embedding
# vector

In [37]:
def embed(sentence, model="text-embedding-3-small"):
    resp = client.embeddings.create(
        model=model,
        input=sentence
    )
    return resp.data[0].embedding

v1 = embed("Forxiga caused UTI in 8.4% of patients") 
v2 = embed("Dapagliflozin leads to urinary infections")

In [38]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    return dot_product / (norm_a * norm_b)

similarity = cosine_similarity(v1, v2)
print(f"Cosine Similarity: {similarity:.4f}")

Cosine Similarity: 0.5360


In [39]:
Q = [0.9, 0.8, 0.7] # user questions: 
A = [0.8, 0.7, 0.6]
B = [0.7, 0.6, 0.5]

cosine_similarities = {
    "Q_A": cosine_similarity(Q, A),
    "Q_B": cosine_similarity(Q, B),
}

cosine_similarities


{'Q_A': np.float64(0.9998962099324299), 'Q_B': np.float64(0.9994375175330728)}

In [40]:
forxiga_chunks = [
    "Forxiga caused UTI in 8.4% of patients vs 5.2% in placebo group.",
    "Forxiga reduces HbA1c by 0.89% vs placebo (p<0.001) in DECLARE-TIMI 58.",
    "Forxiga is contraindicated in patients with eGFR below 45 mL/min.",
    "AstraZeneca Q3 revenue grew 12% year over year.",  # irrelevant chunk
]

chunk_embeddings = [embed(chunk) for chunk in forxiga_chunks]
forxiga_vectors = np.array(chunk_embeddings, dtype="float32")

import faiss
dim = 1536
M = 32 # number of neighbors
index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.ef_search = 64
index.add(forxiga_vectors)


In [41]:
index.ntotal

4

In [42]:
query = "AstraZeneca revenue?"
q_vec = np.array([embed(query)], dtype="float32") 

D,I = index.search(q_vec, k=2)

for i in I[0]:
    print(forxiga_chunks[i])

AstraZeneca Q3 revenue grew 12% year over year.
Forxiga reduces HbA1c by 0.89% vs placebo (p<0.001) in DECLARE-TIMI 58.


In [53]:
# import chromadb

# client = chromadb.Client()
# col = client.create_collection("forxigadb")


In [44]:

forxiga_chunks = [
    "Forxiga caused UTI in 8.4% of patients vs 5.2% in placebo group.",
    "Forxiga reduces HbA1c by 0.89% vs placebo in DECLARE-TIMI 58 trial.",
    "Forxiga is contraindicated in patients with eGFR below 45 mL/min.",
    "Forxiga dose is 10mg once daily with or without food.",
    "AstraZeneca Q3 revenue grew 12% year over year.",   # irrelevant chunk
]

col.add(documents=forxiga_chunks,
ids=[str(i) for i in range(len(forxiga_chunks))]) #all-MiniLM-L6-v2


In [45]:
results = col.query(
    query_texts=["UTI percentage Forxiga"],
    n_results=2
)
results

{'ids': [['0', '2']],
 'embeddings': None,
 'documents': [['Forxiga caused UTI in 8.4% of patients vs 5.2% in placebo group.',
   'Forxiga is contraindicated in patients with eGFR below 45 mL/min.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.40674257278442383, 1.050191044807434]]}

In [ ]:
# Assignments
# Explore chromaDB



In [54]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter # chunk 
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

In [83]:
# Text
fake_forxiga_text = """
Forxiga (dapagliflozin) is an SGLT2 inhibitor approved for Type 2 Diabetes.
It works by blocking glucose reabsorption in the kidneys.

Clinical Efficacy:
Forxiga reduces HbA1c by 0.89% versus placebo in the DECLARE-TIMI 58 trial.
Cardiovascular benefits include 17% reduction in heart failure hospitalization.

Adverse Events:
Forxiga caused urinary tract infections in 8.4% of patients.
Placebo group had UTI rate of 5.2%.
Genital mycotic infections occurred in 6.1% of Forxiga patients.

Renal Considerations:
Forxiga is contraindicated in patients with eGFR below 45 mL/min.
Renal function should be monitored regularly during treatment.

Dosage:
Forxiga dose is 10mg once daily with or without food.
Do not use in Type 1 Diabetes patients.
"""

In [ ]:
document = [Document(
    page_content=fake_forxiga_text,
    metadata={
        "source": "forxiga_clinical_study",
        "date": "2026-01-01",
        "author": "AZ",
    }
)]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20) # fixed chunking
text_chunks = text_splitter.split_documents(document)



In [95]:
chunks

[Document(metadata={}, page_content='Forxiga (dapagliflozin) is an SGLT2 inhibitor approved for Type 2 Diabetes.\nIt works by blocking glucose reabsorption in the kidneys.'),
 Document(metadata={}, page_content='Clinical Efficacy:\nForxiga reduces HbA1c by 0.89% versus placebo in the DECLARE-TIMI 58 trial.\nCardiovascular benefits include 17% reduction in heart failure hospitalization.'),
 Document(metadata={}, page_content='Adverse Events:\nForxiga caused urinary tract infections in 8.4% of patients.\nPlacebo group had UTI rate of 5.2%.\nGenital mycotic infections occurred in 6.1% of Forxiga patients.'),
 Document(metadata={}, page_content='Renal Considerations:\nForxiga is contraindicated in patients with eGFR below 45 mL/min.\nRenal function should be monitored regularly during treatment.'),
 Document(metadata={}, page_content='Dosage:\nForxiga dose is 10mg once daily with or without food.\nDo not use in Type 1 Diabetes patients.')]

In [ ]:
vector_db = Chroma.from_documents(chunks, OpenAIEmbeddings(),persist_directory="./forxiga_vector_db")
retriever = vector_db.as_retriever(search_kwargs={"k": 2})



In [96]:
results = retriever.get_relevant_documents("Who Is MS dhoni?")
results

[Document(id='095c9b47-fc19-474d-a2a6-5a6cedfd6441', metadata={}, page_content='Forxiga (dapagliflozin) is an SGLT2 inhibitor approved for Type 2 Diabetes.\nIt works by blocking glucose reabsorption in the kidneys.'),
 Document(id='d47efebe-fc3e-45dc-ada4-5134f3b54263', metadata={}, page_content='Clinical Efficacy:\nForxiga reduces HbA1c by 0.89% versus placebo in the DECLARE-TIMI 58 trial.\nCardiovascular benefits include 17% reduction in heart failure hospitalization.')]

In [60]:
len(text_chunks[3])

79

In [68]:
from langchain_openai import OpenAIEmbeddings

In [75]:
from langchain_experimental.text_splitter import SemanticChunker
semantic_chunker = SemanticChunker(OpenAIEmbeddings(),breakpoint_threshold_amount=0.3) # semantic chunking

chunks = semantic_chunker.split_text(document.page_content)
chunks

['\nForxiga (dapagliflozin) is an SGLT2 inhibitor approved for Type 2 Diabetes. It works by blocking glucose reabsorption in the kidneys.',
 'Clinical Efficacy:\nForxiga reduces HbA1c by 0.89% versus placebo in the DECLARE-TIMI 58 trial.',
 'Cardiovascular benefits include 17% reduction in heart failure hospitalization.',
 'Adverse Events:\nForxiga caused urinary tract infections in 8.4% of patients.',
 'Placebo group had UTI rate of 5.2%.',
 'Genital mycotic infections occurred in 6.1% of Forxiga patients.',
 'Renal Considerations:\nForxiga is contraindicated in patients with eGFR below 45 mL/min.',
 'Renal function should be monitored regularly during treatment.',
 'Dosage:\nForxiga dose is 10mg once daily with or without food.',
 'Do not use in Type 1 Diabetes patients.',
 '']

In [46]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

In [47]:
# Text
fake_forxiga_text = """
Forxiga (dapagliflozin) is an SGLT2 inhibitor approved for Type 2 Diabetes.
It works by blocking glucose reabsorption in the kidneys.

Clinical Efficacy:
Forxiga reduces HbA1c by 0.89% versus placebo in the DECLARE-TIMI 58 trial.
Cardiovascular benefits include 17% reduction in heart failure hospitalization.

Adverse Events:
Forxiga caused urinary tract infections in 8.4% of patients.
Placebo group had UTI rate of 5.2%.
Genital mycotic infections occurred in 6.1% of Forxiga patients.

Renal Considerations:
Forxiga is contraindicated in patients with eGFR below 45 mL/min.
Renal function should be monitored regularly during treatment.

Dosage:
Forxiga dose is 10mg once daily with or without food.
Do not use in Type 1 Diabetes patients.
"""

In [48]:
documents = [Document(page_content=fake_forxiga_text)]
chunks    = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30).split_documents(documents)
vectordb  = Chroma.from_documents(chunks, OpenAIEmbeddings(), persist_directory="./forxiga_db")
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

results = retriever.get_relevant_documents("What percentage of patients got UTI with Forxiga?")

/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_16025/1423231594.py:6: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents("What percentage of patients got UTI with Forxiga?")


In [ ]:
# RRF_score(doc) = 1/(60 + vector_rank) + 1/(60 + bm25_rank)
# For query "DECLARE-TIMI 58 cardiovascular outcomes Forxiga": # 
# chunk: "DECLARE-TIMI 58 showed 17% reduction in HF hospitalization" # 
# vector_rank=12 (poor — proper noun)
#  bm25_rank=1 (perfect match) 
# # RRF = 1/(60+12) + 1/(60+1) = 0.014 + 0.016 = 0.030 
# # chunk: "Dapagliflozin showed cardiovascular benefits in trial"
#  # vector_rank=2 (good semantic match)
#  bm25_rank=8 (partial match) 
# RRF = 1/(60+2) + 1/(60+8) = 0.016 + 0.015 = 0.031
# wins overall
# Both relevant chunks surface — neither method alone would find both

In [50]:
from pathlib import Path
import json
from langchain_community.document_loaders import PyPDFLoader

bajaj_pdf_dir = Path('./bajaj_finance_pdfs')
pdf_paths = sorted(bajaj_pdf_dir.glob('*.pdf'))
metadata_path = bajaj_pdf_dir / 'metadata.json'
metadata_by_filename = {}

if metadata_path.exists():
    metadata_by_filename = {
        item['filename']: item
        for item in json.loads(metadata_path.read_text(encoding='utf-8'))
    }

bajaj_docs = []
for pdf_path in pdf_paths:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    extra_metadata = metadata_by_filename.get(pdf_path.name, {})

    for page in pages:
        page.metadata.update(extra_metadata)
        page.metadata['source'] = str(pdf_path)

    bajaj_docs.extend(pages)

len(pdf_paths), len(bajaj_docs), bajaj_docs[0].metadata


/Users/rahultiwari/Documents/ml_akila/ml-akila/lib/python3.13/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


(10,
 20,
 {'producer': 'PyPDF',
  'creator': 'PyPDF',
  'creationdate': '2026-03-25T18:36:03+00:00',
  'source': 'bajaj_finance_pdfs/01_personal_loan_eligibility.pdf',
  'total_pages': 2,
  'page': 0,
  'page_label': '1',
  'filename': '01_personal_loan_eligibility.pdf',
  'topic': 'Personal Loan Eligibility Criteria',
  'department': 'sales',
  'loan_type': 'personal',
  'year': 2024})

In [51]:
bajaj_chunks = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
).split_documents(bajaj_docs)

bajaj_vectordb = Chroma.from_documents(
    documents=bajaj_chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory='./bajaj_finance_db'
)

bajaj_retriever = bajaj_vectordb.as_retriever(search_kwargs={'k': 3})
len(bajaj_chunks)


51

In [52]:
query = 'What is the minimum CIBIL score for a personal loan and what are the interest rate slabs?'
bajaj_results = bajaj_retriever.invoke(query)

for doc in bajaj_results:
    print(doc.metadata.get('filename'), '|', doc.metadata.get('topic'))
    print(doc.page_content[:500])
    print('-' * 80)


04_home_loan_eligibility.pdf | Home Loan Eligibility and Terms
3. **MINIMUM INCOME REQUIREMENTS**
- For salaried individuals: Minimum annual income of Rs. 3,00,000
- For self-employed individuals: Minimum annual income of Rs. 4,00,000
4. **AGE LIMITS**
- Minimum age: 23 years
- Maximum age at loan maturity: 65 years for salaried individuals and 70 years for self-employed individuals
5. **TENURE OPTIONS**
- Minimum tenure: 5 years
- Maximum tenure: 30 years
6. **INTEREST RATE SLABS BASED ON CIBIL SCORE**
- CIBIL score of 750 and above: Starting interest rate
--------------------------------------------------------------------------------
02_personal_loan_interest_rates.pdf | Personal Loan Interest Rates and Charges
BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT
PERSONAL LOAN INTEREST RATES AND CHARGES
Department: Risk  |  Category: Personal Loan  |  Year: 2024  |  CONFIDENTIAL
BAJAJ FINANCE LIMITED
PERSONAL LOAN INTEREST RATES AND CHARGES POLICY
EFFECTIVE DATE: [INSERT DATE]
PURPOSE:
